# PRACTICE 2 - PREDICTING PRODUCT SALES

## Notebook báo cáo cuối

Notebook tổng hợp toàn bộ nội dung đã thực hiện: phân tích yêu cầu, lựa chọn và thu thập dữ liệu, làm sạch, feature engineering, chia dữ liệu theo thời gian, huấn luyện và so sánh mô hình, hình ảnh sau train, lựa chọn model tốt nhất và giao diện Streamlit.

> **Trạng thái:** Report-only notebook - số liệu và hình ảnh được lấy từ lần chạy pipeline thật, không phải dữ liệu minh họa.

## 0. Tóm tắt kết quả

- Dữ liệu raw: **73.100 dòng**, 15 cột, từ `2022-01-01` đến `2024-01-01`.
- Dữ liệu modeling: **70.300 dòng** sau khi tạo đặc trưng lịch sử 28 ngày.
- Bài toán: hồi quy dự đoán `Units Sold` theo cửa hàng - sản phẩm - ngày.
- Mô hình được so sánh: Inventory-only Linear, Linear Regression, Ridge, Decision Tree, Random Forest, HistGradientBoosting, Extra Trees và baseline lag-7.
- Mô hình tốt nhất: **Inventory-only Linear Regression**, được chọn bằng RMSE validation.
- Kết quả test: **RMSE 87,859**, **MAE 68,833**, **MSE 7.719,167**, **R² 0,34032**.
- So với baseline lag-7, RMSE giảm khoảng **42,2%**.
- Sản phẩm cuối: model `joblib`, schema dự đoán, metrics, 9 hình báo cáo và giao diện Streamlit.

## 1. Define Problem - Xác định bài toán

Một công ty thương mại điện tử/bán lẻ muốn dự đoán số lượng sản phẩm bán được để hỗ trợ quản lý tồn kho và lập kế hoạch marketing.

| Thành phần | Định nghĩa trong project |
| --- | --- |
| Loại bài toán | Supervised regression trên panel time-series |
| Đơn vị quan sát | Một cửa hàng - một sản phẩm - một ngày |
| Target | `units_sold` - số đơn vị bán được trong ngày |
| Input | Giá, giảm giá, tồn kho, lượng đặt, danh mục, vùng, thời tiết, holiday/promotion, mùa vụ và lịch sử bán |
| Mục tiêu chọn model | RMSE nhỏ nhất, đồng thời theo dõi MAE, MSE và R² |
| Ràng buộc | Dự đoán cuối không âm; không dùng thông tin tương lai |

Dữ liệu ở cấp cửa hàng - sản phẩm - ngày, không có khóa khách hàng. Vì vậy project không tự tạo nhân khẩu học khách hàng hoặc Customer Lifetime Value.

## 2. Yêu cầu Practice 2 và phạm vi đã thực hiện

| Yêu cầu | Cách thực hiện | Dẫn chứng |
| --- | --- | --- |
| Data preparation | Kiểm tra schema, kiểu dữ liệu, dòng trùng và miền hợp lệ | `src/prepare_data.py` |
| Categorical encoding | One-hot encoding trong `ColumnTransformer` | `src/train.py` |
| Train/test split | Chia train/validation/test theo thời gian | `cleaning_report.json` |
| Feature engineering | Đặc trưng lịch, lag và rolling theo store-product | `modeling_data.csv` |
| Model selection | So sánh bảy model ML và một baseline | `model_comparison.csv` |
| Hyperparameter tuning | Chọn tham số trên validation | `training_summary.json` |
| Evaluation | MAE, MSE, RMSE, R² trên temporal test | `best_model.json` |
| Deployment | Giao diện Python bằng Streamlit | `app.py` |
| Hình ảnh | Xuất EDA, model comparison, prediction, residual và importance | `reports/figures/` |

## 3. Lựa chọn và thu thập dữ liệu

Nguồn chính: **Kaggle - Retail Store Inventory Forecasting Dataset**.

- Dataset ref: `anirudhchauhan/retail-store-inventory-forecasting-dataset`.
- File: `retail_store_inventory.csv`.
- Giấy phép: **CC0 - Public Domain**.
- Pipeline tải ZIP bằng Kaggle public dataset API, giải nén CSV và xác minh schema.
- Metadata lưu URL, thời điểm tải, kích thước và SHA-256 tại `data/raw/source_metadata.json`.

Nguồn này được chọn vì có target số lượng bán và các yếu tố liên quan trực tiếp đến tồn kho, giá, promotion, thời tiết và mùa vụ. Một nguồn Kaggle khác đã được kiểm tra nhưng file thực tế không khớp mô tả marketing/mùa vụ nên không được sử dụng.

## 4. Dữ liệu thô

Dữ liệu có 73.100 dòng, tương ứng 731 ngày × 5 cửa hàng × 20 sản phẩm. Không phát hiện missing hoặc dòng trùng trong bản raw đã tải.

| Nhóm | Các trường chính |
| --- | --- |
| Thời gian | `Date` |
| Sản phẩm/cửa hàng | `Store ID`, `Product ID`, `Category`, `Region` |
| Vận hành | `Inventory Level`, `Units Ordered` |
| Giá/marketing | `Price`, `Discount`, `Holiday/Promotion`, `Competitor Pricing` |
| Bối cảnh | `Weather Condition`, `Seasonality` |
| Target | `Units Sold` |
| Trường loại khỏi model | `Demand Forecast` |

`Demand Forecast` là một dự báo đã có sẵn trong nguồn nên bị loại khỏi feature để tránh target leakage.

## 5. Làm sạch dữ liệu

Quy trình làm sạch gồm:

1. Chuẩn hóa tên cột sang `snake_case` và parse `date`.
2. Ép kiểu các trường số, kiểm tra missing và dòng trùng.
3. Kiểm tra ràng buộc: target không âm, giá dương, discount trong 0-100, tồn kho/lượng đặt không âm.
4. Giữ lại các peak bán hàng hợp lệ, không xóa chỉ vì là statistical outlier.
5. Tạo lag/rolling rồi loại 2.800 dòng đầu chuỗi chưa đủ 28 ngày lịch sử.

| Chỉ tiêu | Kết quả |
| --- | ---: |
| Raw rows | 73.100 |
| Duplicates removed | 0 |
| Invalid rows removed | 0 |
| Rows thiếu lịch sử 28 ngày | 2.800 |
| Processed rows | 70.300 |

## 6. Phân tích dữ liệu và ảnh trước/sau train

### Hình 1 - Phân phối target

![Phân phối số lượng bán](../reports/figures/target_distribution.png)

Target lệch phải: phần lớn bản ghi có doanh số thấp/trung bình, một số ngày đạt gần 500 đơn vị.

### Hình 2 - Doanh số theo thời gian

![Doanh số theo tuần](../reports/figures/sales_over_time.png)

### Hình 3 - Doanh số theo danh mục

![Doanh số theo danh mục](../reports/figures/sales_by_category.png)

### Hình 4 - Ảnh hưởng của holiday/promotion

![Ảnh hưởng của holiday và promotion](../reports/figures/promotion_effect.png)

## 7. Preprocessing và Feature Engineering

Pipeline sử dụng `scikit-learn Pipeline` và `ColumnTransformer` để preprocessing khi train và inference giống nhau.

- Numeric: median imputation; Linear Regression dùng thêm StandardScaler.
- Categorical: most-frequent imputation và one-hot encoding với `handle_unknown='ignore'`.
- Lịch: year, month, quarter, ISO week, day of week, weekend, đầu/cuối tháng và sin/cos chu kỳ.
- Lịch sử bán: lag 1/7/14/28 ngày, rolling mean/std 7 và 28 ngày.

Mọi đặc trưng lịch sử được tính riêng theo `store_id + product_id` và dùng `shift(1)`. Vì vậy target của ngày hiện tại không xuất hiện trong feature của chính ngày đó. Tổng cộng có **32 feature** được đưa vào model.

## 8. Chia dữ liệu theo thời gian

Không dùng random split vì các bản ghi có thứ tự thời gian. Tất cả ngày train đứng trước validation và test.

| Tập | Số dòng | Khoảng ngày | Số ngày |
| --- | ---: | --- | ---: |
| Train | 49.200 | 2022-01-29 → 2023-06-04 | 492 |
| Validation | 10.500 | 2023-06-05 → 2023-09-17 | 105 |
| Test | 10.600 | 2023-09-18 → 2024-01-01 | 106 |

Hyperparameter được chọn trên validation. Sau đó cấu hình thắng của từng họ model được fit lại trên train + validation và đánh giá một lần trên test.

## 9. Các model đã train

| Model | Vai trò | Tham số được thử/chọn |
| --- | --- | --- |
| Seasonal lag-7 | Baseline | Dự đoán bằng doanh số 7 ngày trước |
| Inventory-only Linear | Model tuyến tính lọc nhiễu | Chỉ dùng `inventory_level` |
| Linear Regression | Baseline ML tuyến tính đầy đủ | One-hot + scaling |
| Ridge Regression | Tuyến tính có regularization | `alpha` 100/1.000/10.000 |
| Decision Tree | Quan hệ phi tuyến đơn cây | `max_depth`, `min_samples_leaf` |
| Random Forest | Ensemble bagging | `max_depth`, `min_samples_leaf`, 120 cây |
| HistGradientBoosting | Boosting phi tuyến | `learning_rate`, `max_leaf_nodes`, 180 iterations |
| Extra Trees | Ensemble cây ngẫu nhiên hóa | `max_depth`, `min_samples_leaf`, 160 cây |

Random seed được cố định là 42. Các model cùng sử dụng một feature set và một temporal split để việc so sánh công bằng.

## 10. So sánh kết quả các model

| Model | MAE test | MSE test | RMSE test | R² test |
| --- | ---: | ---: | ---: | ---: |
| **Inventory-only Linear** | **68,833** | **7.719,167** | **87,859** | **0,34032** |
| Ridge Regression | 68,887 | 7.726,326 | 87,900 | 0,33971 |
| Linear Regression | 68,890 | 7.726,998 | 87,903 | 0,33965 |
| HistGradientBoosting | 68,928 | 7.761,023 | 88,097 | 0,337 |
| Extra Trees | 69,073 | 7.794,562 | 88,287 | 0,33388 |
| Random Forest | 70,364 | 7.937,884 | 89,095 | 0,322 |
| Decision Tree | 70,830 | 8.380,871 | 91,547 | 0,284 |
| Seasonal lag-7 baseline | 118,576 | 23.099,081 | 151,984 | -0,974 |

Inventory-only Linear có validation RMSE và test RMSE tốt nhất. Việc bỏ các feature gần như không có tín hiệu làm R² test tăng từ 0,33965 lên 0,34032 và giảm nhẹ RMSE/MAE. Mức tăng nhỏ nhưng hợp lệ, không dùng `demand_forecast`.

### Hình 5 - So sánh RMSE của các model

![So sánh RMSE các model](../reports/figures/model_comparison_rmse.png)

Bảy model ML đều vượt baseline lag-7 rõ rệt. Inventory-only Linear giảm RMSE khoảng 42,2% so với baseline.

### Hình 6 - Model an toàn và benchmark có nguy cơ leakage

![So sánh model an toàn và leakage benchmark](../reports/figures/safe_vs_leakage_r2.png)

`Demand Forecast` cho R² khoảng 0,9918 nhưng tương quan 0,9969 với target và sai số chỉ khoảng 8-10 đơn vị. Đây là near-direct target proxy trong dữ liệu synthetic, vì vậy chỉ được báo cáo để kiểm toán và không được dùng trong model/UI.

## 11. Đánh giá dự đoán và residual

### Hình 7 - Actual vs Predicted

![Actual vs Predicted](../reports/figures/actual_vs_predicted.png)

Model bám được xu hướng tăng của target nhưng co dự đoán về vùng trung tâm; các ngày doanh số rất cao vẫn khó dự đoán chính xác.

### Hình 8 - Phân phối residual

![Phân phối residual](../reports/figures/residual_distribution.png)

### Hình 9 - Residual theo thời gian test

![Residual theo thời gian](../reports/figures/residuals_over_time.png)

Residual trung bình theo ngày dao động quanh 0 và không thể hiện drift một chiều kéo dài trong giai đoạn test.

### Hình 10 - Độ ổn định sai số theo tháng test

![Độ ổn định model theo tháng](../reports/figures/monthly_model_stability.png)

RMSE theo tháng nằm trong khoảng xấp xỉ 85,8-88,8 đơn vị. Không có tháng nào suy giảm đột biến so với toàn bộ test.

## 12. Feature Importance

### Hình 11 - Permutation importance của model tốt nhất

![Permutation importance](../reports/figures/feature_importance.png)

`inventory_level` là feature có đóng góp lớn nhất trong dữ liệu này. Các feature còn lại có mức cải thiện nhỏ hơn nhiều. Đây là đặc điểm của bộ dữ liệu synthetic và cần được kiểm chứng lại nếu áp dụng trên dữ liệu doanh nghiệp thật.

## 13. Lựa chọn model tốt nhất

Model cuối được chọn là **Inventory-only Linear Regression** và được lưu tại `models/best_model.joblib`. Model được chọn bằng RMSE validation, không dùng test để chọn họ model.

Lý do chọn:

- RMSE test thấp nhất: 87,859.
- MAE test thấp nhất: 68,833.
- R² test cao nhất trong bảy model ML: 0,34032.
- Model đơn giản, inference nhanh và dễ giải thích.
- Pipeline đã đóng gói preprocessing cùng model, giảm sai khác giữa train và dự đoán.

Dự đoán đầu ra được chặn dưới tại 0 vì số lượng bán không thể âm. Quantile 90% của absolute residual validation là khoảng 152,1 đơn vị và được dùng làm khoảng dự đoán tham khảo trong UI.

## 14. Giao diện Python sau train

Giao diện được viết bằng Streamlit trong `app.py`. App chỉ nạp model đã train, không train lại khi mở.

Các nhóm input:

- Ngày dự đoán, store, product, category và region.
- Tồn kho, lượng đặt, giá, discount và giá đối thủ.
- Thời tiết, holiday/promotion và seasonality.
- Lag/rolling được tính từ lịch sử trước ngày dự đoán.

Lệnh chạy:

```powershell
streamlit run app.py
```

Smoke test đã xác nhận Streamlit trả HTTP 200 trên localhost.

## 15. Cách tái tạo kết quả và artifact

```powershell
python -m pip install -r requirements.txt
python -m src.run_pipeline
python -m compileall -q app.py src tests
python -m pytest -q
streamlit run app.py
```

| Artifact | Vai trò |
| --- | --- |
| `data/raw/source_metadata.json` | Nguồn, license, thời điểm tải và SHA-256 |
| `data/processed/cleaning_report.json` | Kết quả làm sạch và temporal split |
| `reports/metrics/model_comparison.csv` | Metrics của toàn bộ model |
| `reports/metrics/training_summary.json` | Cấu hình và thông tin training |
| `reports/metrics/feature_signal_audit.json` | Correlation và kiểm toán tín hiệu feature |
| `reports/metrics/leakage_benchmark.json` | Benchmark `demand_forecast`, không deploy |
| `reports/metrics/test_period_stability.csv` | Sai số theo tháng test |
| `reports/figures/` | 11 ảnh báo cáo sau train |
| `models/best_model.joblib` | Pipeline model tốt nhất |
| `models/feature_schema.json` | Schema input cho UI |

## 16. Kết luận và giới hạn

Project đã thực hiện trọn vẹn quy trình Practice 2 và một vòng cải tiến: chẩn đoán trần dữ liệu, thêm model/regularization, chọn model bằng validation, kiểm toán leakage, theo dõi ổn định theo tháng và bổ sung khoảng dự đoán 90% cho UI.

Giới hạn cần nêu khi trình bày:

- Dataset là synthetic, không đại diện trực tiếp cho một doanh nghiệp thật.
- Không có dữ liệu khách hàng cá nhân hoặc marketing spend chi tiết.
- R² test khoảng 0,34 cho thấy còn nhiều biến động chưa được giải thích.
- `inventory_level` chi phối mạnh kết quả; cần kiểm tra định nghĩa thời điểm đo khi chuyển sang dữ liệu thực để tránh leakage nghiệp vụ.
- UI yêu cầu lag/rolling đã được tính từ lịch sử gần nhất.

Notebook này là report-only: **19 markdown cells, 0 code cells**, sử dụng đúng metrics và ảnh đã lưu từ pipeline hiện tại.